[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%207/L13_Cobalt_Support_Copilot.ipynb)

# Cobalt Support Copilot — Part 1: the layer that **reads**
### ISBA 2411 · Week 7 · Lecture 13

---

### The situation

You just became the product manager for support at **Cobalt**, a B2B analytics platform.

- **~5,000 tickets a week.** Three support agents.
- Roughly **40%** are variations of the same handful of questions.
- Agents spend the first minutes of every ticket just working out *what it's about* and *who should own it*.

Nobody has ever labelled this data. There is no training set. There is no budget for one this quarter.

**Today we build the part of the product that reads the inbox** — search it, sort it, and decide
what can be handled automatically versus what needs a human. Next lecture we make it *act*.

---

### How to use this notebook

You will **not** write code from scratch. Every cell is already written — you just run it.

- 🔮 **Before you run this, predict:** — stop and commit to an answer. This is the part that makes it stick.
- 🎛️ **YOUR KNOB** — one thing you change, then re-run, and watch what happens.
- ✅ **What just happened** — the plain-language explanation.
- 💼 **At work this means** — why anyone would pay you for this.

### ⚠️ First: switch on the GPU

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**

Without it the model cells take several minutes instead of several seconds.

In [ ]:
%pip install -q transformers sentence-transformers sentencepiece scikit-learn pandas matplotlib

import pandas as pd, numpy as np, matplotlib.pyplot as plt, torch, textwrap
pd.set_option("display.max_colwidth", 90)

DEVICE = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available(), "\n")

tickets = pd.read_csv("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_tickets.csv")
print(f"{len(tickets):,} tickets loaded")
tickets[["ticket_id","created_at","plan","priority","ticket_text"]].head()

---
## Act 0 · The inbox you just inherited

Before any technology, look at the problem. This is the whole job: **5,000 of these a week, and nobody
has told you what any of them are about.**

Notice there is **no category column** in what we loaded. That is the honest starting position for
almost every real text project — you have the text, and nothing else.

In [ ]:
# What does the raw inbox actually look like?
for t in tickets.ticket_text.sample(6, random_state=7):
    print(" •", textwrap.shorten(t, 95))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
tickets.priority.value_counts().reindex(["urgent","high","normal","low"]).plot(
    kind="bar", ax=ax[0], color="#4F46E5"); ax[0].set_title("Tickets by priority")
tickets.plan.value_counts().plot(kind="bar", ax=ax[1], color="#059669"); ax[1].set_title("By customer plan")
tickets.ticket_text.str.split().str.len().plot(kind="hist", bins=14, ax=ax[2], color="#B45309")
ax[2].set_title("Ticket length (words)")
for a in ax: a.set_xlabel(""); a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

✅ **What just happened.** Short, messy, human text — a median of about 13 words, no structure,
no labels. A keyword rule would drown in this.

💼 **At work this means:** the first question on any text project is not "which model?" — it's
**"what do I actually have?"** Here: text, and nothing else. That single fact rules out most
traditional analytics and points straight at the technology we're about to use.

---
## Act 1 · Teach the computer to *read*

A computer cannot compare two sentences the way you can. Our first job is turning each ticket into
something comparable — an **embedding**: a list of numbers that stands for the ticket's *meaning*.

Once every ticket is a point in the same space, "find me similar tickets" becomes arithmetic.

🔮 **Before you run this, predict:** We're going to search for **"we are being charged wrong"**.
Not one ticket in this inbox contains that exact phrase. Will the search find the billing complaints anyway?

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
E = encoder.encode(tickets.ticket_text.tolist(), normalize_embeddings=True)
print("every ticket is now a vector:", E.shape, "  (tickets x numbers)")

def keyword_search(q, k=3):
    hits = tickets[tickets.ticket_text.str.contains(q, case=False, na=False)]
    return hits.head(k)

def semantic_search(q, k=3):
    sims = E @ encoder.encode([q], normalize_embeddings=True)[0]
    top = np.argsort(-sims)[:k]
    return tickets.iloc[top].assign(similarity=sims[top].round(2))

QUERY = "we are being charged wrong"
print(f"\nKEYWORD search for {QUERY!r} -> {len(keyword_search(QUERY))} results")
print(f"SEMANTIC search for {QUERY!r}:")
for _, r in semantic_search(QUERY).iterrows():
    print(f"   {r.similarity:.2f}  {textwrap.shorten(r.ticket_text, 88)}")

✅ **What just happened.** Keyword search found **nothing** — no ticket uses that phrasing. Semantic
search returned the billing complaints anyway, including *"We were charged twice this month for the
same seats."* The word "charged" matched, but so did *invoice*, *refund*, *trial* — because the model
places them near each other **by meaning**, not spelling.

💼 **At work this means:** every "search that actually works" feature you've used — help centres,
support tools, internal wikis — is this. It's also the foundation of the retrieval systems we build
in Week 8.

### 🎛️ YOUR KNOB — search the inbox yourself

Change `MY_QUERY` below to anything a support manager might ask, then re-run the cell. Try:
`"customers who want to cancel"` · `"something is broken after your update"` · `"security worries"`

In [ ]:
MY_QUERY = "customers who want to cancel"     # <-- change me, then re-run

for _, r in semantic_search(MY_QUERY, k=5).iterrows():
    print(f"{r.similarity:.2f}  [{r.priority:6}] {textwrap.shorten(r.ticket_text, 85)}")

### The shape of the whole inbox

If every ticket is a point, we can flatten those points onto a page and *look* at the inbox.

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2, random_state=0).fit_transform(E)

plt.figure(figsize=(8.5, 6))
plt.scatter(xy[:,0], xy[:,1], s=34, c="#94A3B8", alpha=.75)
for q, col in [("billing and invoices","#E11D48"), ("cannot sign in","#4F46E5"),
               ("the app is slow","#059669"), ("how do I do this","#B45309")]:
    s = E @ encoder.encode([q], normalize_embeddings=True)[0]
    top = np.argsort(-s)[:14]
    plt.scatter(xy[top,0], xy[top,1], s=52, c=col, label=q)
plt.legend(fontsize=9); plt.xticks([]); plt.yticks([])
plt.title("The inbox, arranged by meaning — neighbourhoods form on their own")
plt.tight_layout(); plt.show()

✅ **What just happened.** Nobody labelled anything, yet tickets about the same thing landed near each
other. **Structure was already in the text** — we just made it visible.

💼 **At work this means:** you can show a leadership team what's in a pile of feedback *before*
spending a dollar on labelling.

---
## Act 2 · Route the inbox — with **zero** labelled examples

Now the product question: **can we sort tickets into the buckets our support team actually uses?**

The traditional answer is "label a few thousand tickets first, then train a classifier." That's weeks
of work and money you don't have.

Instead we'll use **zero-shot classification**: we describe each category *in plain English* and the
model decides which description fits. No training. No labels. It works because the model was already
pretrained on enormous amounts of text — someone else paid for that, and we get it for free.

🔮 **Before you run this, predict:** with **zero** training examples, what accuracy would you bet on?
Random guessing across 8 buckets would be about 12%.

In [ ]:
from transformers import pipeline

# Describe each bucket in plain English. THIS IS THE 'PROGRAM'.
CATEGORIES = {
 "login_access"   : "signing in, passwords, two-factor authentication or account access",
 "billing_plan"   : "an invoice, payment, refund, subscription plan or pricing question",
 "data_sync"      : "a database connector failing to refresh or import data",
 "integrations"   : "connecting to another product such as Slack, Salesforce, webhooks or the API",
 "performance"    : "the product being slow, timing out or hanging",
 "bug_ui"         : "a chart, table or screen displaying something incorrectly",
 "how_to"         : "asking for instructions on how to do something",
 "feature_request": "requesting a new capability that does not exist yet",
}

router = pipeline("zero-shot-classification",
                  model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0", device=DEVICE)

descriptions = list(CATEGORIES.values())
back = {d: k for k, d in CATEGORIES.items()}

out = router(tickets.ticket_text.tolist(), candidate_labels=descriptions, multi_label=False)
routed = tickets.assign(
    predicted =[back[o["labels"][0]] for o in out],
    confidence=[round(o["scores"][0], 2) for o in out],
    second    =[back[o["labels"][1]] for o in out],
)
print("Routed", len(routed), "tickets using zero labelled examples.\n")
routed[["ticket_text","predicted","confidence"]].head(8)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
routed.predicted.value_counts().plot(kind="barh", ax=ax[0], color="#4F46E5")
ax[0].set_title("Where the copilot sent the tickets"); ax[0].invert_yaxis()
ax[1].hist(routed.confidence, bins=18, color="#059669")
ax[1].set_title("How sure was it?"); ax[1].set_xlabel("confidence")
plt.tight_layout(); plt.show()

### Now let's grade it

The dataset *does* have a hidden ground-truth label — we held it back so the routing was honest.
Time to see how a system with **no training data** actually did.

In [ ]:
from sklearn.metrics import accuracy_score

top1 = accuracy_score(tickets.category, routed.predicted)
top2 = np.mean([t in (p, s) for t, p, s in zip(tickets.category, routed.predicted, routed.second)])

print(f"  random guessing        {1/len(CATEGORIES):.1%}")
print(f"  top-1 accuracy         {top1:.1%}   <- the copilot's single best guess")
print(f"  top-2 accuracy         {top2:.1%}   <- correct answer in its top two")
print(f"\n  labelled examples used: 0")
print(f"  models trained:         0")

✅ **What just happened.** From **nothing** — no labels, no training, no GPU-hours — the copilot sorts
the inbox far better than chance, and its top-two suggestions contain the right answer the large
majority of the time.

Read that second number as a product, not a metric: *"show the agent two buttons instead of eight."*

💼 **At work this means:** you can ship a useful v1 **this week**, and only then decide whether
labelling data is worth it. The expensive step is now the *last* resort instead of the first.

### 🎛️ YOUR KNOB — the category descriptions *are* the program

There is no training here, so the only thing you control is **how you describe each bucket**. That
makes wording a design decision, not a detail.

Below, the descriptions have been replaced with lazy one-word versions. Run it and compare the accuracy
to what you just got. Then try writing better ones yourself.

In [ ]:
LAZY = {"login_access":"login", "billing_plan":"billing", "data_sync":"sync",
        "integrations":"integrations", "performance":"performance", "bug_ui":"bug",
        "how_to":"how to", "feature_request":"feature request"}   # <-- rewrite these and re-run

d2 = list(LAZY.values()); back2 = {d: k for k, d in LAZY.items()}
out2 = router(tickets.ticket_text.tolist(), candidate_labels=d2, multi_label=False)
lazy_pred = [back2[o["labels"][0]] for o in out2]

print(f"  careful descriptions : {top1:.1%}")
print(f"  lazy one-word labels : {accuracy_score(tickets.category, lazy_pred):.1%}")
print("\n  Same model. Same tickets. Only the wording changed.")

---
## Act 3 · Where v1 breaks — and what a PM does about it

An average is a bad way to run a product. **Which** tickets does it get wrong, and does that matter?

In [ ]:
recall = (routed.assign(ok=routed.predicted.values == tickets.category.values)
          .groupby(tickets.category).ok.mean().sort_values())

plt.figure(figsize=(8, 3.4))
colors = ["#E11D48" if v < .5 else "#059669" for v in recall.values]
plt.barh(recall.index, recall.values, color=colors)
plt.axvline(.5, ls="--", c="#334155", lw=1)
plt.title("How often each bucket is caught correctly"); plt.xlim(0, 1)
plt.tight_layout(); plt.show()

worst = recall.index[0]
print(f"Worst bucket: {worst}\n")
for _, r in routed[tickets.category.values == worst].head(4).iterrows():
    print(f"  sent to '{r.predicted}' ({r.confidence}) <- {textwrap.shorten(r.ticket_text, 78)}")

✅ **What just happened.** The failures are not random. The weak buckets are the ones whose
**descriptions overlap** — a failing data sync genuinely *is* "something is broken," and an API problem
genuinely *is* "an integration." The model isn't confused; **our categories are.**

💼 **At work this means:** when a classifier underperforms, the first thing to examine is usually your
**taxonomy**, not your model. Categories should describe *what the customer wants done*, not which
internal component is involved.

### 🎛️ YOUR KNOB — the decision that actually ships the product

You do not have to route everything. Route only what the copilot is **sure** about, and send the rest
to a human. That one threshold decides how much work you save and how many mistakes you make.

Move `THRESHOLD` and watch the trade-off.

In [ ]:
THRESHOLD = 0.90        # <-- change me (try 0.5, 0.7, 0.95) and re-run

auto = routed.confidence >= THRESHOLD
acc_auto = accuracy_score(tickets.category[auto], routed.predicted[auto]) if auto.sum() else 0

print(f"  Auto-routed          : {auto.mean():.0%} of the inbox")
print(f"  Accuracy on those    : {acc_auto:.0%}")
print(f"  Sent to a human      : {(~auto).mean():.0%}")
print(f"  Tickets/week handled : ~{int(5000*auto.mean()):,} of 5,000")

grid = np.arange(.3, .99, .02)
cov = [(routed.confidence >= t).mean() for t in grid]
acc = [accuracy_score(tickets.category[routed.confidence >= t],
                      routed.predicted[routed.confidence >= t])
       if (routed.confidence >= t).sum() > 5 else np.nan for t in grid]
plt.figure(figsize=(7.5, 4))
plt.plot(grid, cov, label="share of inbox auto-routed", lw=2.5, color="#4F46E5")
plt.plot(grid, acc, label="accuracy of those decisions", lw=2.5, color="#059669")
plt.axvline(THRESHOLD, ls="--", c="#E11D48", label=f"your threshold ({THRESHOLD})")
plt.xlabel("confidence threshold"); plt.ylim(0, 1); plt.legend(); plt.grid(alpha=.25)
plt.title("The product decision: cover more, or be more right")
plt.tight_layout(); plt.show()

✅ **What just happened.** The two lines move in opposite directions, and **there is no setting that
maximises both.** Where you stand on that curve is a business judgement: how costly is a misrouted
ticket, versus how valuable is an agent's hour?

💼 **At work this means:** this curve — not accuracy — is what you take into the room. "We can
automate 44% of the inbox at 81% accuracy, or 87% of it at 66%" is a decision an executive can
actually make.

---
## What we shipped today

| | |
|---|---|
| **Semantic search** over the whole inbox | finds tickets by *meaning*, not keywords |
| **A router** into 8 buckets | built with **zero** labelled examples |
| **A confidence policy** | auto-handle the easy ones, escalate the rest |
| **Labelling cost so far** | **$0** |

### The one idea to take with you

> Someone else already spent millions of dollars teaching a model to read English.
> **Your job is not to build intelligence — it's to point it at your problem, cheaply, and know where it breaks.**

### Next lecture

Our router is useful but blunt: it needs a human for most of the inbox, and it can't pull out *what
specifically* is broken. Next time we ask what changes when you **do** invest in labelling a few
hundred tickets, how the pretraining that makes all this possible actually worked — and where this
technology confidently makes things up.